# 02 — Exploratory Data Analysis

**Proyecto:** Global Life Expectancy Prediction with Machine Learning  
**Autor:** Marcelo Corro  

Secciones:
1. Setup y carga de datos
2. Vision global — distribucion de esperanza de vida
3. Evolucion temporal 2000–2021
4. Correlaciones con el target
5. Chile vs LATAM vs OCDE
6. Paises destacados
7. Conclusiones

---
## 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

TEMPLATE = 'plotly_white'

COLORES_REGION = {
    'Africa':   '#E63946',
    'Americas': '#457B9D',
    'Asia':     '#F4A261',
    'Europe':   '#2A9D8F',
    'Oceania':  '#8338EC',
    'Otro':     '#ADB5BD',
}

LATAM = ['ARG','BRA','BOL','CHL','COL','CRI','CUB','DOM','ECU',
         'SLV','GTM','HND','HTI','MEX','NIC','PAN','PRY','PER','URY','VEN']

OCDE  = ['AUS','AUT','BEL','CAN','CHL','COL','CRI','CZE','DNK','EST',
         'FIN','FRA','DEU','GRC','HUN','ISL','IRL','ISR','ITA','JPN',
         'KOR','LVA','LTU','LUX','MEX','NLD','NZL','NOR','POL','PRT',
         'SVK','SVN','ESP','SWE','CHE','TUR','GBR','USA']

df = pd.read_csv('data/processed/dataset_final.csv')
print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Paises: {df["iso3"].nunique()} | Anios: {df["anio"].min()}-{df["anio"].max()}')

Dataset cargado: 4070 filas x 21 columnas
Paises: 185 | Anios: 2000-2021


---
## 2. Vision Global — Distribucion de Esperanza de Vida

In [2]:
print('Estadisticas globales de esperanza de vida:')
df['esperanza_vida'].describe().round(2).to_frame()

Estadisticas globales de esperanza de vida:


,esperanza_vida
count,4070.00
mean,70.17
std,8.62
min,40.16
25%,64.48
50%,72.31
75%,76.62
max,84.64


In [3]:
fig = px.box(
    df,
    x='region',
    y='esperanza_vida',
    color='region',
    color_discrete_map=COLORES_REGION,
    points='outliers',
    title='Distribucion de Esperanza de Vida por Region (2000-2021)',
    labels={'esperanza_vida': 'Esperanza de Vida (anos)', 'region': 'Region'},
    template=TEMPLATE,
    category_orders={'region': ['Africa','Americas','Asia','Europe','Oceania','Otro']}
)
fig.update_layout(showlegend=False)
fig.show()

In [4]:
df_2021 = df[df['anio'] == 2021].copy()

fig = px.choropleth(
    df_2021,
    locations='iso3',
    color='esperanza_vida',
    hover_name='pais',
    hover_data={'esperanza_vida': ':.1f', 'iso3': False},
    color_continuous_scale='RdYlGn',
    range_color=[45, 85],
    title='Esperanza de Vida Mundial — 2021',
    labels={'esperanza_vida': 'Esperanza de Vida (anos)'},
    template=TEMPLATE
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

---
## 3. Evolucion Temporal 2000–2021

In [5]:
ev_region = (
    df[df['region'] != 'Otro']
    .groupby(['anio', 'region'])['esperanza_vida']
    .mean()
    .reset_index()
)

fig = px.line(
    ev_region,
    x='anio',
    y='esperanza_vida',
    color='region',
    color_discrete_map=COLORES_REGION,
    markers=True,
    title='Evolucion de Esperanza de Vida por Region (2000-2021)',
    labels={'esperanza_vida': 'Esperanza de Vida (anos)', 'anio': 'Ano', 'region': 'Region'},
    template=TEMPLATE
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [6]:
ganancia = (
    df[df['anio'].isin([2000, 2021])]
    .groupby(['iso3', 'pais', 'region', 'anio'])['esperanza_vida']
    .mean()
    .unstack('anio')
    .dropna()
)
ganancia.columns = ['ev_2000', 'ev_2021']
ganancia['ganancia'] = ganancia['ev_2021'] - ganancia['ev_2000']
ganancia = ganancia.reset_index()

print(f'Ganancia media global 2000-2021: {ganancia["ganancia"].mean():.1f} anos')
print()
print('Top 5 paises con mayor mejora:')
print(ganancia.nlargest(5, 'ganancia')[['pais', 'region', 'ev_2000', 'ev_2021', 'ganancia']].round(1).to_string(index=False))
print()
print('Top 5 paises con menor mejora (o retroceso):')
print(ganancia.nsmallest(5, 'ganancia')[['pais', 'region', 'ev_2000', 'ev_2021', 'ganancia']].round(1).to_string(index=False))

Ganancia media global 2000-2021: 4.3 anos

Top 5 paises con mayor mejora:
    pais region  ev_2000  ev_2021  ganancia
  Rwanda Africa     46.9     67.5      20.6
 Burundi Africa     44.4     64.0      19.6
  Malawi Africa     44.7     62.5      17.8
  Uganda Africa     48.9     66.0      17.1
Ethiopia Africa     50.8     67.8      17.0

Top 5 paises con menor mejora (o retroceso):
       pais   region  ev_2000  ev_2021  ganancia
   Paraguay Americas     74.7     70.4      -4.4
Philippines     Asia     70.1     66.6      -3.6
       Peru Americas     75.1     71.8      -3.4
     Mexico Americas     74.2     70.9      -3.3
       Cuba Americas     76.8     73.8      -3.0


---
## 4. Correlaciones con el Target

In [7]:
features = [
    'esperanza_vida', 'mortalidad_infantil', 'mortalidad_adulta',
    'prevalencia_vih', 'vacunacion_dtp', 'vacunacion_hepatitis_b',
    'medicos_por_10000', 'prevalencia_anemia_ninos', 'pib_per_capita',
    'gasto_salud_pct_pib', 'gasto_educacion_pct_pib', 'acceso_agua_potable',
    'acceso_saneamiento', 'acceso_electricidad', 'tasa_desempleo',
    'urbanizacion_pct', 'usuarios_internet_pct'
]

corr = df[features].corr().round(2)

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Mapa de Correlaciones — Dataset Completo',
    text_auto=True,
    aspect='auto',
    template=TEMPLATE
)
fig.update_layout(height=700)
fig.show()

In [8]:
corr_target = (
    corr['esperanza_vida']
    .drop('esperanza_vida')
    .sort_values(key=abs, ascending=False)
)

fig = px.bar(
    x=corr_target.values,
    y=corr_target.index,
    orientation='h',
    color=corr_target.values,
    color_continuous_scale='RdBu_r',
    range_color=[-1, 1],
    title='Correlacion de cada Feature con Esperanza de Vida',
    labels={'x': 'Correlacion de Pearson', 'y': 'Feature'},
    template=TEMPLATE
)
fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False)
fig.show()

In [9]:
top_features = corr_target.abs().nlargest(4).index.tolist()

fig = make_subplots(rows=2, cols=2, subplot_titles=top_features)

for i, feat in enumerate(top_features):
    row, col = divmod(i, 2)
    for region, grp in df[df['region'] != 'Otro'].groupby('region'):
        x_vals = np.log1p(grp[feat]) if feat == 'pib_per_capita' else grp[feat]
        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=grp['esperanza_vida'],
                mode='markers',
                marker=dict(size=3, color=COLORES_REGION.get(region, '#ADB5BD'), opacity=0.5),
                name=region,
                showlegend=(i == 0),
            ),
            row=row + 1, col=col + 1
        )

x_labels = [f'log({f})' if f == 'pib_per_capita' else f for f in top_features]
for i, label in enumerate(x_labels):
    row, col = divmod(i, 2)
    fig.update_xaxes(title_text=label, row=row + 1, col=col + 1)
    fig.update_yaxes(title_text='Esperanza de Vida', row=row + 1, col=col + 1)

fig.update_layout(
    title='Top 4 Features vs Esperanza de Vida',
    height=600,
    template=TEMPLATE
)
fig.show()

---
## 5. Chile vs LATAM vs OCDE

In [10]:
chile_ev  = df[df['iso3'] == 'CHL'].groupby('anio')['esperanza_vida'].mean()
latam_ev  = df[df['iso3'].isin(LATAM) & (df['iso3'] != 'CHL')].groupby('anio')['esperanza_vida'].mean()
ocde_ev   = df[df['iso3'].isin(OCDE) & (df['iso3'] != 'CHL')].groupby('anio')['esperanza_vida'].mean()
global_ev = df.groupby('anio')['esperanza_vida'].mean()

fig = go.Figure()
for nombre, serie, color, dash in [
    ('Chile',        chile_ev,  '#E63946', 'solid'),
    ('Media LATAM',  latam_ev,  '#457B9D', 'dash'),
    ('Media OCDE',   ocde_ev,   '#2A9D8F', 'dot'),
    ('Media Global', global_ev, '#ADB5BD', 'dashdot'),
]:
    fig.add_trace(go.Scatter(
        x=serie.index, y=serie.values,
        mode='lines+markers',
        name=nombre,
        line=dict(color=color, dash=dash, width=2),
        marker=dict(size=5)
    ))

fig.update_layout(
    title='Esperanza de Vida: Chile vs LATAM vs OCDE vs Global (2000-2021)',
    xaxis_title='Ano',
    yaxis_title='Esperanza de Vida (anos)',
    template=TEMPLATE,
    legend=dict(x=0.01, y=0.01)
)
fig.show()

In [11]:
indicadores_radar = [
    'acceso_agua_potable', 'acceso_electricidad', 'acceso_saneamiento',
    'vacunacion_dtp', 'usuarios_internet_pct', 'gasto_salud_pct_pib'
]

df_2021_comp = df[df['anio'] == 2021]

radar = pd.DataFrame({
    'Chile':      df_2021_comp[df_2021_comp['iso3'] == 'CHL'][indicadores_radar].mean(),
    'LATAM':      df_2021_comp[df_2021_comp['iso3'].isin(LATAM)][indicadores_radar].mean(),
    'OCDE':       df_2021_comp[df_2021_comp['iso3'].isin(OCDE)][indicadores_radar].mean(),
})

fig = go.Figure()
for col, color in [('Chile', '#E63946'), ('LATAM', '#457B9D'), ('OCDE', '#2A9D8F')]:
    fig.add_trace(go.Scatterpolar(
        r=radar[col].values,
        theta=indicadores_radar,
        fill='toself',
        name=col,
        line=dict(color=color)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Radar de Indicadores Clave — Chile vs LATAM vs OCDE (2021)',
    template=TEMPLATE
)
fig.show()

In [12]:
df_2021_rank = df[df['anio'] == 2021].copy()
n_paises = len(df_2021_rank)

print(f'Ranking de Chile entre {n_paises} paises (2021):')
print()
cols_rank = ['esperanza_vida', 'pib_per_capita', 'acceso_agua_potable',
             'acceso_electricidad', 'medicos_por_10000', 'usuarios_internet_pct']

for col in cols_rank:
    rank = df_2021_rank[col].rank(ascending=False, method='min')
    pos  = int(rank[df_2021_rank['iso3'] == 'CHL'].values[0])
    val  = float(df_2021_rank[df_2021_rank['iso3'] == 'CHL'][col].values[0])
    print(f'  {col:<28} -> puesto {pos:>3} / {n_paises}  (valor: {val:.1f})')

Ranking de Chile entre 185 paises (2021):

  esperanza_vida               -> puesto  31 / 185  (valor: 79.0)
  pib_per_capita               -> puesto  51 / 185  (valor: 16216.2)
  acceso_agua_potable          -> puesto  60 / 185  (valor: 98.8)
  acceso_electricidad          -> puesto   1 / 185  (valor: 100.0)
  medicos_por_10000            -> puesto  55 / 185  (valor: 29.8)
  usuarios_internet_pct        -> puesto  35 / 185  (valor: 90.2)


---
## 6. Paises Destacados

In [13]:
top10    = df_2021_rank.nlargest(10, 'esperanza_vida')[['pais', 'region', 'esperanza_vida']].round(1)
bottom10 = df_2021_rank.nsmallest(10, 'esperanza_vida')[['pais', 'region', 'esperanza_vida']].round(1)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Top 10 — Mayor Esperanza de Vida', 'Bottom 10 — Menor Esperanza de Vida'])

fig.add_trace(go.Bar(
    x=top10['esperanza_vida'],
    y=top10['pais'],
    orientation='h',
    marker_color='#2A9D8F',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bottom10['esperanza_vida'],
    y=bottom10['pais'],
    orientation='h',
    marker_color='#E63946',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Esperanza de Vida por Pais — 2021',
    height=400,
    template=TEMPLATE
)
fig.update_yaxes(autorange='reversed')
fig.show()

In [14]:
fig = px.scatter(
    ganancia,
    x='ev_2000',
    y='ev_2021',
    color='region',
    color_discrete_map=COLORES_REGION,
    hover_name='pais',
    hover_data={'ganancia': ':.1f', 'region': False},
    title='Esperanza de Vida 2000 vs 2021 por Pais',
    labels={'ev_2000': 'Esperanza de Vida 2000', 'ev_2021': 'Esperanza de Vida 2021'},
    template=TEMPLATE
)

lim = [ganancia[['ev_2000', 'ev_2021']].min().min() - 1,
       ganancia[['ev_2000', 'ev_2021']].max().max() + 1]
fig.add_shape(type='line', x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
              line=dict(dash='dash', color='gray', width=1))

chl = ganancia[ganancia['iso3'] == 'CHL'].iloc[0]
fig.add_annotation(x=chl['ev_2000'], y=chl['ev_2021'], text='Chile',
                   showarrow=True, arrowhead=2, ax=30, ay=-20, font=dict(color='#E63946'))

fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

---
## 7. Conclusiones del EDA

### Hallazgos principales

**Distribucion global:**
- La esperanza de vida mundial promedio es **70.2 anos** (2000-2021).
- Europa y Oceania lideran con medias sobre **77 anos**. Africa tiene la media mas baja (~60 anos).
- El rango es amplio: de ~40 anos (paises sub-saharianos en crisis) a ~85 anos (Japon, paises nordicos).

**Evolucion temporal:**
- Todas las regiones mejoraron entre 2000 y 2021.
- Africa tuvo la mayor ganancia proporcional (efecto de programas VIH/SIDA y vacunacion).
- Se observa una leve caida global en 2020-2021 por efecto COVID.

**Features mas correlacionadas con el target:**
- Correlacion negativa fuerte: mortalidad infantil, mortalidad adulta, prevalencia VIH.
- Correlacion positiva fuerte: acceso a electricidad, saneamiento, agua potable, internet, PIB per capita.
- El PIB per capita tiene una relacion no lineal con el target (log-transformacion recomendada).

**Chile:**
- Consistentemente por encima de la media LATAM y cerca de la media OCDE.
- Lidera en acceso a agua potable y electricidad dentro de LATAM.
- La caida de 2020-2021 es visible y coincide con el impacto de la pandemia.

**Siguiente paso:** `notebooks/03_modeling.ipynb` — Entrenamiento y evaluacion de modelos ML